# Download readable articles and books

Files are saved directly to `~/Downloads/books_articles`. There is no database and no image download.

- arXiv: search XML and paper PDF
- Europe PMC / PubMed Central: open-access article XML and extracted plain text
- Open Library: readable plain-text books only when the search result has a public Internet Archive scan

The code keeps SSL certificate verification enabled.

In [6]:
from pathlib import Path
import time
from urllib.parse import quote, urlencode
from urllib.request import Request, urlopen
import json
import re
import ssl
import xml.etree.ElementTree as ET

try:
    import certifi
except ImportError as error:
    raise RuntimeError('Missing certificate bundle. Run: %pip install certifi') from error

DOWNLOAD_FOLDER = Path.home() / 'Downloads' / 'AgenticFolder' / 'project0001' / 'a.Extractions' /  'a.Raw Extraction Documents - Species RAG'
DOWNLOAD_FOLDER.mkdir(parents=True, exist_ok=True)
USER_AGENT = 'BooksArticlesDownloader/0.5 (contact: your-email@example.com)'
SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())

# 60s was too generous for what should be fast JSON/text responses -- a single
# slow/stalled request could block a loop for a full minute. 15s is enough
# headroom for a normal response while failing fast on a stalled connection.
REQUEST_TIMEOUT_SECONDS = 15

def safe_filename(text: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', text).strip('._')[:120]

def download(url: str, filename: str, params: dict | None = None, quiet: bool = False) -> Path:
    if params:
        url += ('&' if '?' in url else '?') + urlencode(params)
    destination = DOWNLOAD_FOLDER / filename
    request = Request(url, headers={'User-Agent': USER_AGENT})
    with urlopen(request, context=SSL_CONTEXT, timeout=REQUEST_TIMEOUT_SECONDS) as response, destination.open('wb') as file:
        while chunk := response.read(1024 * 1024):
            file.write(chunk)
    if not quiet:
        print(f'Saved: {destination}')
    return destination

def read_json(url: str, params: dict) -> dict:
    request = Request(url + '?' + urlencode(params), headers={'User-Agent': USER_AGENT, 'Accept': 'application/json'})
    with urlopen(request, context=SSL_CONTEXT, timeout=REQUEST_TIMEOUT_SECONDS) as response:
        return json.load(response)

def xml_to_text(xml_file: Path, text_file: Path) -> Path:
    root = ET.parse(xml_file).getroot()
    text = '\n'.join(line.strip() for line in root.itertext() if line.strip())
    text_file.write_text(text, encoding='utf-8')
    print(f'Saved: {text_file}')
    return text_file

def write_source_metadata(text_path: Path, **fields) -> None:
    """Write a sidecar '<text_path>.meta.json' capturing real citation data
    resolved during download (title, url, isbn, doi, etc.) -- so downstream
    RAG ingestion (04_ChunkEmbedChromaRetrieve) can build proper citations
    instead of guessing from the filename.

    Only pass fields you actually know are true; leave unknown ones out
    (or None) rather than guessing -- a wrong citation is worse than a
    missing one.
    """
    meta_path = text_path.with_suffix(text_path.suffix + '.meta.json')
    payload = {k: v for k, v in fields.items() if v is not None}
    meta_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')


In [7]:
# Corpus manisfest
from corpus_manifest import BOOKS, SPECIES_WIKI, REGIONAL_STATUS_SOURCES, PAPERS, ALL_DOCUMENTS, DOCUMENT_GROUPS

# Count the number of documents listed
print(f"ALL_DOCUMENTS {len(ALL_DOCUMENTS)}")
for label, docs in DOCUMENT_GROUPS.items():
    print(label, len(docs))

# Get the title of documents
def get_unique_values_titles(documents, key='title'):
    return [doc[key] for doc in documents]

REQUEST_DELAY_SECONDS = 1.0  # politeness delay between API calls in each loop below

# Report titles no found
def report(api_name: str, status: str, detail: str) -> None:
    """Consistent status line for every extraction loop, e.g. '[arXiv] NO MATCH: <title>'."""
    print(f'[{api_name}] {status}: {detail}')


ALL_DOCUMENTS 460
species_wiki 298
regional_status 34
books 65
papers 52
videos 11


In [ ]:
### arXiv: for each paper in PAPERS, download its PDF.

ATOM_NS = {'atom': 'http://www.w3.org/2005/Atom'}
API_NAME = 'arXiv'

def arxiv_search_first_id(title: str) -> str | None:
    request = Request(
        'https://export.arxiv.org/api/query?' + urlencode({
            'search_query': f'ti:"{title}"',
            'start': 0,
            'max_results': 1,
        }),
        headers={'User-Agent': USER_AGENT},
    )
    with urlopen(request, context=SSL_CONTEXT, timeout=60) as response:
        root = ET.fromstring(response.read())
    entry = root.find('atom:entry', ATOM_NS)
    if entry is None:
        return None
    raw_id = entry.find('atom:id', ATOM_NS).text  # e.g. http://arxiv.org/abs/2401.00001v1
    return raw_id.rsplit('/', 1)[-1].split('v')[0]

def crossref_doi_for_title(title: str) -> str | None:
    """Resolve the best-matching DOI for a title via Crossref -- used when no doi is known."""
    request = Request(
        'https://api.crossref.org/works?' + urlencode({'query.bibliographic': title, 'rows': 1}),
        headers={'User-Agent': USER_AGENT},
    )
    with urlopen(request, context=SSL_CONTEXT, timeout=60) as response:
        data = json.load(response)
    items = data.get('message', {}).get('items', [])
    return items[0]['DOI'] if items else None

def arxiv_search_by_doi(doi: str) -> str | None:
    request = Request(
        'https://export.arxiv.org/api/query?' + urlencode({
            'search_query': f'doi:"{doi}"',
            'start': 0,
            'max_results': 1,
        }),
        headers={'User-Agent': USER_AGENT},
    )
    with urlopen(request, context=SSL_CONTEXT, timeout=60) as response:
        root = ET.fromstring(response.read())
    entry = root.find('atom:entry', ATOM_NS)
    if entry is None:
        return None
    raw_id = entry.find('atom:id', ATOM_NS).text
    return raw_id.rsplit('/', 1)[-1].split('v')[0]

for doc in PAPERS:
    title = doc['title']
    destination = DOWNLOAD_FOLDER / f'arxiv_{safe_filename(title)}.pdf'
    if destination.exists():
        report(API_NAME, 'SKIP', destination.name)
        continue
    try:
        arxiv_id = doc.get('arxiv_id')
        via = 'manifest arxiv_id' if arxiv_id else None

        if arxiv_id is None:
            arxiv_id = arxiv_search_first_id(title)

        if arxiv_id is None:
            doi = doc.get('doi')
            if doi:
                arxiv_id = arxiv_search_by_doi(doi)
                via = 'manifest doi' if arxiv_id else None
            else:
                doi = crossref_doi_for_title(title)
                if doi:
                    arxiv_id = arxiv_search_by_doi(doi)
                    via = f'resolved doi {doi}' if arxiv_id else None

        if arxiv_id is None:
            report(API_NAME, 'NO MATCH', title)
            continue

        if via:
            report(API_NAME, 'FALLBACK', f'{title} -> matched via {via}')
        download(f'https://arxiv.org/pdf/{arxiv_id}', destination.name)
        write_source_metadata(
            destination,
            source_type='arxiv',
            title=title,
            arxiv_id=arxiv_id,
            url=f'https://arxiv.org/abs/{arxiv_id}',
            doi=doc.get('doi') or locals().get('doi'),
            category=doc.get('category'),
        )
    except Exception as error:
        report(API_NAME, 'FAILED', f'{title} ({error})')
    time.sleep(REQUEST_DELAY_SECONDS)


[arXiv] NO MATCH: Biological Flora of the British Isles: Fallopia japonica
[arXiv] NO MATCH: Impacts of Himalayan balsam (Impatiens glandulifera) on riparian plant communities in the UK
[arXiv] NO MATCH: A review of the ecology and control of Rhododendron ponticum
[arXiv] NO MATCH: Genetics of hybridisation between native bluebell (Hyacinthoides non-scripta) and Spanish bluebell (H. hispanica)
[arXiv] NO MATCH: Coniine and other poisonous alkaloids in Conium maculatum (hemlock): chemistry and toxicology
[arXiv] NO MATCH: Cardiac glycosides in Digitalis purpurea (foxglove): biosynthesis and pharmacological history
[arXiv] NO MATCH: Tropane alkaloid poisoning by Atropa belladonna and Datura stramonium: clinical review
[arXiv] NO MATCH: Amanita phalloides-Associated Liver Failure: Molecular Mechanisms and Management
[arXiv] NO MATCH: Molecular phylogeny and reclassification of the genus Amanita in Europe
[arXiv] NO MATCH: Orellanine poisoning from Cortinarius rubellus and C. orellanus: de

In [ ]:
# Open Library: for each book in BOOKS, save readable OCR text from Internet Archive.

API_NAME = 'Open Library'
MAX_FALLBACK_DOCS = 8
MAX_ISBNS_PER_DOC = 3
FALLBACK_TIME_BUDGET_SECONDS = 20

def significant_words(text: str) -> set[str]:
    return {w for w in re.findall(r'[a-z]+', text.lower()) if len(w) > 3}

def isbn_to_ocaid(isbn: str) -> str | None:
    try:
        edition = read_json(f'https://openlibrary.org/isbn/{isbn}.json', {})
    except Exception:
        return None
    return edition.get('ocaid')

def isbn_fallback_identifier(title: str, docs: list[dict]) -> str | None:
    '''Fallback path: use an ISBN found IN THE SEARCH RESULTS (not the manifest) to find a scan.
    Bounded by MAX_FALLBACK_DOCS, MAX_ISBNS_PER_DOC, and FALLBACK_TIME_BUDGET_SECONDS
    so one stubborn title can never block the whole loop.'''
    deadline = time.monotonic() + FALLBACK_TIME_BUDGET_SECONDS
    for result_doc in docs[:MAX_FALLBACK_DOCS]:
        if time.monotonic() > deadline:
            report(API_NAME, 'FALLBACK', f'{title} -> ISBN fallback time budget exceeded, moving on')
            return None
        isbns = (result_doc.get('isbn') or [])[:MAX_ISBNS_PER_DOC]
        matched_title = result_doc.get('title', '')
        if not isbns or not (significant_words(title) & significant_words(matched_title)):
            continue
        for isbn in isbns:
            if time.monotonic() > deadline:
                report(API_NAME, 'FALLBACK', f'{title} -> ISBN fallback time budget exceeded, moving on')
                return None
            ocaid = isbn_to_ocaid(isbn)
            if ocaid:
                return ocaid
    return None

for doc in BOOKS:
    title = doc['title']
    text_filename = f'open_library_{safe_filename(title)}.txt'
    text_path = DOWNLOAD_FOLDER / text_filename
    if text_path.exists():
        report(API_NAME, 'SKIP', text_filename)
        continue
    try:
        identifier = None
        via = None

        manifest_isbn = doc.get('isbn')
        if manifest_isbn:
            identifier = isbn_to_ocaid(manifest_isbn)
            via = 'manifest isbn' if identifier else None

        if identifier is None:
            search_results = read_json(
                'https://openlibrary.org/search.json',
                {'q': title, 'limit': 50, 'fields': 'title,author_name,ia,public_scan_b,isbn'},
            )
            docs_found = search_results.get('docs', [])
            book = next((item for item in docs_found if item.get('public_scan_b') and item.get('ia')), None)
            if book is not None:
                matched_title = book.get('title', '')
                if significant_words(title) & significant_words(matched_title):
                    identifier = book['ia'][0]

            if identifier is None:
                identifier = isbn_fallback_identifier(title, docs_found)
                via = 'isbn found in search results' if identifier else None

        if identifier is None:
            report(API_NAME, 'NO MATCH', title)
            continue

        item = read_json(f'https://archive.org/metadata/{quote(identifier, safe="")}', {})
        text_file = next((file['name'] for file in item.get('files', []) if file.get('name', '').endswith('_djvu.txt')), None)
        if text_file is None:
            report(API_NAME, 'NO MATCH', f'no readable text for {identifier}')
            continue

        if via:
            report(API_NAME, 'FALLBACK', f'{title} -> matched via {via} ({identifier})')
        download(
            f'https://archive.org/download/{quote(identifier, safe="")}/{quote(text_file)}',
            text_filename,
            quiet=True,
        )
        # archive.org's own /metadata endpoint often has real creator/date
        # fields for the scanned item -- genuine IA-supplied data, not a
        # guess, so it's safe to carry through as citation metadata.
        ia_meta = item.get('metadata', {})
        write_source_metadata(
            text_path,
            source_type='open_library',
            title=title,
            archive_org_identifier=identifier,
            url=f'https://archive.org/details/{identifier}',
            isbn=manifest_isbn,
            archive_org_creator=ia_meta.get('creator'),
            archive_org_date=ia_meta.get('date'),
            archive_org_publisher=ia_meta.get('publisher'),
            manifest_notes=doc.get('notes'),
            category=doc.get('category'),
        )
    except Exception as error:
        report(API_NAME, 'FAILED', f'{title} ({error})')
    time.sleep(REQUEST_DELAY_SECONDS)


[Open Library] SKIP: open_library_On_the_Origin_of_Species.txt
[Open Library] SKIP: open_library_The_Voyage_of_the_Beagle.txt
[Open Library] SKIP: open_library_The_Malay_Archipelago.txt
[Open Library] SKIP: open_library_The_Natural_History_of_Selborne.txt
[Open Library] SKIP: open_library_English_Botany.txt
[Open Library] SKIP: open_library_The_Insect_World_of_J._Henri_Fabre.txt
[Open Library] SKIP: open_library_Outlines_of_British_Fungology.txt
[Open Library] SKIP: open_library_A_Manual_of_British_Botany.txt
[Open Library] SKIP: open_library_The_Birds_of_Great_Britain.txt
[Open Library] SKIP: open_library_Curtis_s_Botanical_Magazine.txt
[Open Library] SKIP: open_library_British_Entomology_John_Curtis.txt
[Open Library] SKIP: open_library_A_History_of_British_Birds_William_Yarrell.txt
[Open Library] SKIP: open_library_Wild_Flowers.txt
[Open Library] SKIP: open_library_The_Fungus_Flora_of_Yorkshire.txt
[Open Library] SKIP: open_library_The_Flowering_Plants_of_Great_Britain.txt
[Open Lib

In [ ]:
# Wikipedia: for each species in SPECIES_WIKI, save the extracted plain-text summary.

API_NAME = 'Wikipedia'

def wikipedia_best_title(query: str) -> str | None:
    search_results = read_json(
        'https://en.wikipedia.org/w/api.php',
        {'action': 'query', 'list': 'search', 'srsearch': query, 'srlimit': 1, 'format': 'json'},
    )
    matches = search_results.get('query', {}).get('search', [])
    return matches[0]['title'] if matches else None

def strip_parenthetical(text: str) -> str:
    return re.sub(r'\s*\([^)]*\)', '', text).strip()

for doc in SPECIES_WIKI:
    query = doc['title']
    text_path = DOWNLOAD_FOLDER / f'wikipedia_{safe_filename(query)}.txt'
    if text_path.exists():
        report(API_NAME, 'SKIP', text_path.name)
        continue
    try:
        page_title = doc.get('wikipedia_title')
        via = 'manifest wikipedia_title' if page_title else None

        if page_title is None:
            page_title = wikipedia_best_title(query)

        simplified = None
        if page_title is None:
            simplified = strip_parenthetical(query)
            if simplified and simplified != query:
                page_title = wikipedia_best_title(simplified)
                via = f'simplified name "{simplified}"' if page_title else None

        if page_title is None:
            report(API_NAME, 'NO MATCH', query)
            continue

        extract_result = read_json(
            'https://en.wikipedia.org/w/api.php',
            {
                'action': 'query',
                'prop': 'extracts',
                'explaintext': 1,
                'titles': page_title,
                'format': 'json',
                'redirects': 1,
            },
        )
        pages = extract_result.get('query', {}).get('pages', {})
        page = next(iter(pages.values()), None)
        extract_text = (page or {}).get('extract', '').strip()
        if not extract_text:
            report(API_NAME, 'NO MATCH', f'no readable text for {page_title}')
            continue

        if via:
            report(API_NAME, 'FALLBACK', f'{query} -> matched via {via}')
        text_path.write_text(extract_text, encoding='utf-8')
        # page.get('title') is MediaWiki's own resolved canonical title
        # (redirects already followed), so it's more reliable than page_title
        # for building the URL when the two differ.
        resolved_title = (page or {}).get('title') or page_title
        write_source_metadata(
            text_path,
            source_type='wikipedia',
            title=resolved_title,
            url='https://en.wikipedia.org/wiki/' + quote(resolved_title.replace(' ', '_')),
            query_title=query,
            category=doc.get('category'),
        )
    except Exception as error:
        report(API_NAME, 'FAILED', f'{query} ({error})')
    time.sleep(REQUEST_DELAY_SECONDS)


[Wikipedia] SKIP: wikipedia_Japanese_knotweed_Reynoutria_japonica.txt
[Wikipedia] SKIP: wikipedia_Himalayan_balsam_Impatiens_glandulifera.txt
[Wikipedia] SKIP: wikipedia_Giant_hogweed_Heracleum_mantegazzianum.txt
[Wikipedia] SKIP: wikipedia_Rhododendron_ponticum.txt
[Wikipedia] SKIP: wikipedia_New_Zealand_pigmyweed_Crassula_helmsii.txt
[Wikipedia] SKIP: wikipedia_Floating_pennywort_Hydrocotyle_ranunculoides.txt
[Wikipedia] SKIP: wikipedia_Parrot_s_feather_Myriophyllum_aquaticum.txt
[Wikipedia] SKIP: wikipedia_Water_fern_fairy_fern_Azolla_filiculoides.txt
[Wikipedia] SKIP: wikipedia_American_skunk_cabbage_Lysichiton_americanus.txt
[Wikipedia] SKIP: wikipedia_Cherry_laurel_Prunus_laurocerasus.txt
[Wikipedia] SKIP: wikipedia_Cotoneaster_various_spp.txt
[Wikipedia] SKIP: wikipedia_Spanish_bluebell_Hyacinthoides_hispanica.txt
[Wikipedia] SKIP: wikipedia_Water_primrose_Ludwigia_grandiflora.txt
[Wikipedia] SKIP: wikipedia_Fanwort_Cabomba_caroliniana.txt
[Wikipedia] SKIP: wikipedia_Curly_water